In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('DateFruit_Dataset.csv')

In [3]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [4]:
df.shape

(898, 35)

In [5]:
df.isnull().sum()

AREA             0
PERIMETER        0
MAJOR_AXIS       0
MINOR_AXIS       0
ECCENTRICITY     0
EQDIASQ          0
SOLIDITY         0
CONVEX_AREA      0
EXTENT           0
ASPECT_RATIO     0
ROUNDNESS        0
COMPACTNESS      0
SHAPEFACTOR_1    0
SHAPEFACTOR_2    0
SHAPEFACTOR_3    0
SHAPEFACTOR_4    0
MeanRR           0
MeanRG           0
MeanRB           0
StdDevRR         0
StdDevRG         0
StdDevRB         0
SkewRR           0
SkewRG           0
SkewRB           0
KurtosisRR       0
KurtosisRG       0
KurtosisRB       0
EntropyRR        0
EntropyRG        0
EntropyRB        0
ALLdaub4RR       0
ALLdaub4RG       0
ALLdaub4RB       0
Class            0
dtype: int64

In [6]:
X = df.drop('Class', axis=1)
y = df["Class"]

y.unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [7]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
le = LabelEncoder()

y = le.fit_transform(y)


In [8]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

# ANN

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset

In [11]:
X_train_tesnor = torch.tensor(X_train_scaled, dtype = torch.float32)
y_train_tesnor = torch.tensor(y_train, dtype=torch.long)

X_test_tesnor = torch.tensor(X_test_scaled, dtype = torch.float32)
y_test_tensor = torch.tensor(y_test, dtype = torch.long)


In [12]:
train_dataset = TensorDataset(X_train_tesnor, y_train_tesnor)
test_dataset = TensorDataset(X_test_tesnor, y_test_tensor)

In [13]:
train_loader = DataLoader(train_dataset,batch_size=32, shuffle = True)
test_loader = DataLoader(test_dataset,batch_size=32)

In [14]:
# Building our ANN Model

class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()
        
        self.model = nn.Sequential(
            # input layer
            nn.Linear(X.shape[1],64),
            nn.ReLU(),
            # hidden layer
            nn.Linear(64, 64),
            nn.ReLU(),
            # output layer
            nn.Linear(64, 7)
        )
    def forward(self,x):
        return self.model(x)

In [15]:
# create our model

model = ANN()
# loss & optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())


In [16]:
#Training the NN

epochs = 100
for epoch in range(epochs):
    model.train()
    
    running_loss= 0.0
    
    for xb,yb in train_loader:
        optimizer.zero_grad()
        
        outputs = model(xb)
        
        loss = criterion(outputs, yb)
        
        loss.backward()
        
        optimizer.step()  # learning the parameters  
        running_loss += loss.item()
    
    train_loss = running_loss/len(train_loader)
    
    print(f"epoch = {epoch+1}/{epochs}, loss = {train_loss}")
    
    


epoch = 1/100, loss = 1.7192325643871142
epoch = 2/100, loss = 1.1593276210453198
epoch = 3/100, loss = 0.7801020378651826
epoch = 4/100, loss = 0.580870711285135
epoch = 5/100, loss = 0.4625728979058888
epoch = 6/100, loss = 0.4056266688782236
epoch = 7/100, loss = 0.35544434708097705
epoch = 8/100, loss = 0.3155733742143797
epoch = 9/100, loss = 0.28897931718307995
epoch = 10/100, loss = 0.2609473048992779
epoch = 11/100, loss = 0.24297621457473093
epoch = 12/100, loss = 0.230589689767879
epoch = 13/100, loss = 0.2235620941804803
epoch = 14/100, loss = 0.20504774315201718
epoch = 15/100, loss = 0.19631650104470874
epoch = 16/100, loss = 0.18596337136367094
epoch = 17/100, loss = 0.18378035205861795
epoch = 18/100, loss = 0.18098323695037677
epoch = 19/100, loss = 0.16773005089034204
epoch = 20/100, loss = 0.15910064186091008
epoch = 21/100, loss = 0.15778127604204675
epoch = 22/100, loss = 0.1534698999122433
epoch = 23/100, loss = 0.15182911863793497
epoch = 24/100, loss = 0.14060406

In [17]:
# evaluate 

model.eval()
total = 0
correct = 0

with torch.no_grad():
    for xb,yb in test_loader:
        outputs = model(xb)
        _, predicted = torch.max(outputs, 1)
        total += yb.size(0) # actual samples in each batch
        correct += (predicted == yb).sum().item()
        
        
print("total_values:",total)
print("corect :", correct)

print("Accuracy:", correct/total * 100)

total_values: 180
corect : 169
Accuracy: 93.88888888888889


In [23]:
# applying pca

from sklearn.decomposition import PCA
pca = PCA(n_components=0.99)

X_pca = pca.fit_transform(X_train_scaled)
print(X_pca)

print("number of features after PCA:", X_pca.shape[1])
print("number of principal components:", pca.n_components_)

[[-0.28415332  1.23992508 -0.90211146 ... -0.30287286 -0.53332348
   0.31334617]
 [-6.99046075 -1.76942012 -1.01542427 ...  0.44034606 -0.13048339
  -0.06619484]
 [ 3.70225162 -4.91050434  3.46131414 ... -0.12828413 -0.13642352
  -0.5810119 ]
 ...
 [ 0.35819055 -2.12527292  0.89201255 ...  0.03486556  0.04946645
  -0.40448106]
 [-2.41851829  2.93123663  0.20163661 ... -0.14491035  0.0826858
   0.19100905]
 [ 3.1926709  -0.97453667  1.0655243  ... -0.18894625  0.03507756
   0.0242918 ]]
number of features after PCA: 15
number of principal components: 15


In [ ]:
print("Explained variance ratio:", pca.explained_variance_ratio_)

Explained variance ratio: [0.40992337 0.22985566 0.11387629 0.06257129 0.04991463 0.03679119
 0.02674211 0.01828489 0.014797  ]


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV


param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [5, 10, 20, 30, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}
grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)
y_pred = grid_search.predict(X_test_scaled)
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification report:\n", classification_report(y_test, y_pred))

Best parameters: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best cross-validation score: 0.8370920745920746
Accuracy: 0.7777777777777778
Classification report:
               precision    recall  f1-score   support

           0       1.00      0.50      0.67        12
           1       0.48      0.65      0.55        20
           2       1.00      0.76      0.86        50
           3       0.47      0.80      0.59        10
           4       0.91      0.83      0.87        35
           5       0.94      1.00      0.97        33
           6       0.52      0.65      0.58        20

    accuracy                           0.78       180
   macro avg       0.76      0.74      0.73       180
weighted avg       0.83      0.78      0.79       180

